<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Exercises_XP_MCP_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Minimal MCP over STDIO (Student)

Build a tiny MCP server and client that talk over STDIO. This code is supposed to be executed in a local jupyter notebook not Colab's notebook.

## What you'll learn
- How MCP structures hosts/clients/servers and why STDIO is great locally.
- How to register a tool (action) and a resource (read-only context) on a server.
- How to write a client that initializes, lists, and invokes those features.

## Setup
Run the install cell, then restart the runtime if Colab asks. Python 3.10+ required.

In [9]:
# Install MCP CLI + SDK
%pip install -qU "mcp[cli]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 14.4 MB/s eta 0:00:00


In [10]:
# Quick verify
!python --version
!mcp --help | head -n 5

Python 3.12.13
                                                                                
 Usage: mcp [OPTIONS] COMMAND [ARGS]...                                         
                                                                                
 MCP development tools                                                          
                                                                                


## A. Server (server.py)
Create a small MCP server named "Demo" with:
- Tool `add(a: int, b: int) -> int` returning the sum.
- Resource template `greeting://{name}` returning "Hello, {name}!".
- Start the STDIO loop in `__main__`.

In [11]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

# FastMCP is a high-level framework that makes creating MCP servers simple.
# We initialize our server and give it a name.
mcp = FastMCP("Demo")

# A 'tool' is a function that the server provides for the client to execute.
# This tool takes two integers and returns their sum.
@mcp.tool()
def add(a: int, b: int) -> int:
    """Return the sum of two integers."""
    # This print will show up in the server logs when the tool is called
    print(f"[Server] Adding {a} + {b}")
    return a + b

# A 'resource' is a piece of data or context that the client can read.
# The URI template 'greeting://{name}' allows the client to request dynamic content.
@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a greeting for the given name."""
    print(f"[Server] Preparing greeting for: {name}")
    return f"Hello, {name}!"

if __name__ == "__main__":
    # This starts the server using Standard Input/Output (STDIO).
    # It waits for commands from a connected client.
    mcp.run()

Overwriting server.py


## B. Client (client.py)
Write a client that:
1) Spawns the server via STDIO using the MCP CLI.
2) Initializes a session.
3) Lists resources and tools, printing their names.
4) Reads `greeting://hello` and prints it.
5) Calls tool `add` with a=1, b=7 and prints the result.

In [16]:
%%writefile client.py
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# This configuration tells the client how to start the server process.
# We use the 'mcp run' command to execute our server.py file.
server_params = StdioServerParameters(
    command="mcp",
    args=["run", "server.py"],
    env=None
)

def extract_content(payload):
    """
    Helper function to simplify the output.
    MCP responses often contain metadata; we just want the 'text' content.
    """
    if hasattr(payload, "contents") and payload.contents:
        first = payload.contents[0]
        if hasattr(first, "text"): return first.text
    return str(payload)

async def run():
    # Step 1: Establish a communication pipe (STDIO) with the server
    async with stdio_client(server_params) as (read, write):
        # Step 2: Open an MCP session over that pipe
        async with ClientSession(read, write) as session:
            # Initialize the connection handshake
            await session.initialize()
            print("--- Client Connected to MCP Server ---\n")

            # Step 3: Ask the server for a list of available resources
            resources = await session.list_resources()
            print("1. Available Resources:", [r.uri for r in resources.resources])

            # Step 4: Ask the server for a list of available tools
            tools = await session.list_tools()
            print("2. Available Tools:", [t.name for t in tools.tools])

            # Step 5: Read a specific resource (the greeting)
            print("\n3. Fetching resource 'greeting://hello'...")
            greeting = await session.read_resource("greeting://hello")
            print("Server says:", extract_content(greeting))

            # Step 6: Use the 'add' tool provided by the server
            print("\n4. Requesting tool 'add' with values 1 and 7...")
            result = await session.call_tool("add", arguments={"a": 1, "b": 7})
            print("Server calculation result:", extract_content(result))

if __name__ == "__main__":
    # asyncio.run is the entry point for running asynchronous Python code
    asyncio.run(run())

Overwriting client.py


## C. Run
One terminal (client spawns server):
```
python client.py
```

Or two terminals:
```
mcp run server.py
python client.py
```

In Colab, run the next cell (client will spawn the server automatically).

In [17]:
!python client.py

--- Client Connected to MCP Server ---

[07/10/26 23:20:00] INFO     Processing request of type            ]8;id=103902;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=939041;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListResourcesRequest                               
Step 1: Resources found -> []
                    INFO     Processing request of type            ]8;id=96914;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=611162;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   
Step 2: Tools found -> ['add']

Step 3: Reading resource 'greeting://hello'...
                    INFO     Processing request of type            ]8;id=323785;file:///usr/local/lib/python3.12/dist-packa

## Troubleshooting
- `mcp: command not found` ? rerun the install cell or restart runtime.
- Connection closed ? open a second terminal and run `mcp run server.py` to check server errors.
- Type errors ? ensure JSON args are ints for `add`.